In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes
!pip install torchvision pillow

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from datasets import load_dataset

dataset = load_dataset("rootsautomation/RICO-Screen2Words")
dataset


In [ ]:
small_dataset = dataset["train"].shuffle(seed=42).select(range(2000))

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

model_name = "Salesforce/blip-image-captioning-base"

processor = BlipProcessor.from_pretrained(model_name)
model = BlipForConditionalGeneration.from_pretrained(model_name)

In [ ]:
def preprocess(example):

    caption = example["captions"]

    # captions sometimes come as a list → convert to string
    if isinstance(caption, list):
        caption = caption[0]

    inputs = processor(
        images=example["image"],
        text=caption,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    inputs["labels"] = inputs["input_ids"]

    # remove batch dimension
    inputs = {k: v.squeeze() for k, v in inputs.items()}

    return inputs


processed_dataset = small_dataset.map(
    preprocess,
    remove_columns=small_dataset.column_names
)

In [ ]:
print(small_dataset[0]["captions"])
print(type(small_dataset[0]["captions"]))

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],   # correct modules for BLIP
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

In [ ]:
print(small_dataset[0]["captions"])
print(type(small_dataset[0]["captions"]))

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
)

trainer.train()

In [ ]:
model.save_pretrained("rico-blip-lora")
processor.save_pretrained("rico-blip-lora")

In [ ]:
model.push_to_hub("rico-ui-captioner")
processor.push_to_hub("rico-ui-captioner")


In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import requests
from io import BytesIO

model = BlipForConditionalGeneration.from_pretrained("surabhimuralidhar/rico-ui-captioner")
processor = BlipProcessor.from_pretrained("surabhimuralidhar/rico-ui-captioner")

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/coco_sample.png"

image = Image.open(BytesIO(requests.get(url).content)).convert("RGB")

inputs = processor(images=image, return_tensors="pt")

output = model.generate(**inputs)

caption = processor.decode(output[0], skip_special_tokens=True)

print("Generated Caption:", caption)

In [ ]:
from IPython.display import display

display(small_dataset[0]["image"])
print("Actual Caption:", small_dataset[0]["captions"])

In [ ]:
image = small_dataset[0]["image"]

inputs = processor(images=image, return_tensors="pt")

output = model.generate(**inputs)

caption = processor.decode(output[0], skip_special_tokens=True)

print("Generated Caption:", caption)

In [ ]:
from IPython.display import display

sample = small_dataset[5]

display(sample["image"])

print("Ground Truth:", sample["captions"])

inputs = processor(images=sample["image"], return_tensors="pt")

output = model.generate(**inputs)

pred = processor.decode(output[0], skip_special_tokens=True)

print("Model Prediction:", pred)

In [ ]:
from peft import PeftModel

base_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model = PeftModel.from_pretrained(
    base_model,
    "surabhimuralidhar/rico-ui-captioner"
)

# merge LoRA weights
model = model.merge_and_unload()

In [ ]:
model.save_pretrained("rico-final-model")
processor.save_pretrained("rico-final-model")

In [ ]:
model.push_to_hub("rico-ui-captioner-final")
processor.push_to_hub("rico-ui-captioner-final")

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import requests
from io import BytesIO

# Hugging Face model repo
MODEL_ID = "surabhimuralidhar/rico-ui-captioner-final"

print("Loading model from Hugging Face...")

# Load model and processor
model = BlipForConditionalGeneration.from_pretrained(MODEL_ID)
processor = BlipProcessor.from_pretrained(MODEL_ID)

print("Model loaded successfully!")

# Sample image for testing
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/coco_sample.png"

print("Downloading sample image...")

image = Image.open(BytesIO(requests.get(url).content)).convert("RGB")

# Preprocess image
inputs = processor(images=image, return_tensors="pt")

print("Running inference...")

# Generate caption
output = model.generate(**inputs)

caption = processor.decode(output[0], skip_special_tokens=True)

print("\nGenerated Caption:")
print(caption)